In [ ]:
# 1. Config - only edit this cell
import os, time
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

ROOT = "/data/noise_project_v2"                       # folder with clean_500/, noisy_500_v2/, noisy_metadata_v2.csv
OLD_ROOT = "/content/drive/MyDrive/NIST/noise_project"  # prefix stored inside the csv (replaced by ROOT)
OUT = "/data/outputs/svdnet"                          # checkpoints + logs + enhanced images
os.makedirs(OUT, exist_ok=True)

MODE, SIZE = "resize", 256     # "resize" or "crop"
BATCH, EPOCHS, LR = 10, 60, 1e-4
DELTA, ADV_W, SVD_T = 0.84, 0.01, 0.9
RESUME = False                 # True -> continue from OUT/last.pt
device = torch.device("cuda")

In [ ]:
# 2. Data: metadata + leakage check
df = pd.read_csv(f"{ROOT}/noisy_metadata_v2.csv")
for c in ["clean_path", "noisy_path"]:
    df[c] = df[c].str.replace(OLD_ROOT, ROOT, regex=False)

print(df["split"].value_counts())
print("missing clean:", (~df["clean_path"].apply(os.path.exists)).sum(),
      "| missing noisy:", (~df["noisy_path"].apply(os.path.exists)).sum())

s = {k: set(df[df.split == k]["participant"]) for k in ["train", "val", "test"]}
print("leakage train&val / train&test / val&test:",
      len(s["train"] & s["val"]), len(s["train"] & s["test"]), len(s["val"] & s["test"]))

In [ ]:
# 3. Dataset + loaders
class FPDataset(Dataset):
    def __init__(self, df, split, mode="resize", size=256):
        self.d = df[df.split == split].reset_index(drop=True)
        self.split, self.mode, self.size = split, mode, size

    def __len__(self):
        return len(self.d)

    def __getitem__(self, i):
        r = self.d.iloc[i]
        noisy = Image.open(r["noisy_path"]).convert("L")
        clean = Image.open(r["clean_path"]).convert("L")
        if self.mode == "resize":
            noisy = noisy.resize((self.size, self.size), Image.BILINEAR)
            clean = clean.resize((self.size, self.size), Image.BILINEAR)
        else:  # crop: random for train, center for val/test
            w, h = noisy.size
            s = self.size
            if self.split == "train":
                x = np.random.randint(0, max(1, w - s + 1))
                y = np.random.randint(0, max(1, h - s + 1))
            else:
                x, y = max(0, (w - s) // 2), max(0, (h - s) // 2)
            noisy = noisy.crop((x, y, x + s, y + s))
            clean = clean.crop((x, y, x + s, y + s))
        return TF.to_tensor(noisy), TF.to_tensor(clean)   # [0,1], 1 channel

train_dl = DataLoader(FPDataset(df, "train", MODE, SIZE), batch_size=BATCH, shuffle=True,
                      num_workers=4, pin_memory=True, drop_last=True)
val_dl  = DataLoader(FPDataset(df, "val", MODE, SIZE),  batch_size=BATCH, num_workers=4)
test_dl = DataLoader(FPDataset(df, "test", MODE, SIZE), batch_size=BATCH, num_workers=4)

x, y = next(iter(train_dl))
print(x.shape, y.shape, x.min().item(), x.max().item())

In [ ]:
# 4. Model: SVD layer + generator + discriminator
class SVDLayer(nn.Module):
    """Keeps top-k channel components that explain t of the energy."""
    def __init__(self, t=0.9):
        super().__init__()
        self.t = t

    def forward(self, x):
        B, C, H, W = x.shape
        with torch.autocast("cuda", enabled=False):
            X = x.reshape(B, C, H * W).float()
            with torch.no_grad():
                ev, U = torch.linalg.eigh(X @ X.transpose(1, 2))
                ev, U = ev.flip(-1).clamp(min=0), U.flip(-1)
                cum = ev.cumsum(-1) / (ev.sum(-1, keepdim=True) + 1e-8)
                k = (cum < self.t).sum(-1) + 1
                mask = (torch.arange(C, device=x.device)[None] < k[:, None]).float()
                U = U * mask[:, None, :]
            Xd = U @ (U.transpose(1, 2) @ X)
        return Xd.reshape(B, C, H, W).to(x.dtype)

class Generator(nn.Module):
    def __init__(self, t=0.9):
        super().__init__()
        self.c1, self.c2, self.c3 = nn.Conv2d(1, 64, 3, 1, 1), nn.Conv2d(64, 128, 3, 1, 1), nn.Conv2d(128, 256, 3, 1, 1)
        self.d1, self.d2 = nn.ConvTranspose2d(256, 256, 3, 1, 1), nn.ConvTranspose2d(256, 128, 3, 1, 1)
        self.d3, self.d4 = nn.ConvTranspose2d(128, 64, 3, 1, 1), nn.ConvTranspose2d(64, 1, 3, 1, 1)
        self.svd = nn.ModuleList([SVDLayer(t) for _ in range(3)])

    def forward(self, z):
        e1 = self.svd[0](torch.tanh(self.c1(z)))
        e2 = self.svd[1](torch.tanh(self.c2(e1)))
        e3 = self.svd[2](torch.tanh(self.c3(e2)))
        d1 = torch.tanh(self.d1(e3))
        d2 = torch.tanh(self.d2(d1)) + e2      # skip
        d3 = torch.tanh(self.d3(d2)) + e1      # skip
        return torch.sigmoid(self.d4(d3))      # [0,1]

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        ch = [1, 64, 128, 256, 128, 64]
        L = []
        for i in range(5):
            L += [nn.Conv2d(ch[i], ch[i + 1], 3, 1, 1), nn.ReLU(True)]
            if i in (1, 3):
                L.append(nn.MaxPool2d(2))
        self.f = nn.Sequential(*L, nn.AdaptiveAvgPool2d(4))
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(64 * 16, 512), nn.ReLU(True), nn.Linear(512, 2))

    def forward(self, x):
        return self.fc(self.f(x))   # logits (CrossEntropy applies softmax)

G, D = Generator(SVD_T).to(device), Discriminator().to(device)
print(f"G {sum(p.numel() for p in G.parameters())/1e6:.2f}M | D {sum(p.numel() for p in D.parameters())/1e6:.2f}M params")

In [ ]:
# 5. Training setup + evaluation helper
!pip install -q pytorch-msssim
from pytorch_msssim import ms_ssim, ssim
from torch.amp import GradScaler, autocast

optG = torch.optim.Adam(G.parameters(), lr=LR)
optD = torch.optim.Adam(D.parameters(), lr=LR)
scG, scD = GradScaler("cuda"), GradScaler("cuda")
ce = nn.CrossEntropyLoss()
torch.backends.cudnn.benchmark = True

@torch.no_grad()
def evaluate(dl):
    G.eval(); mse = ps = ss = n = 0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        with autocast("cuda"):
            p = G(x)
        p = p.float()
        m = F.mse_loss(p, y, reduction="none").mean((1, 2, 3))
        mse += m.sum().item()
        ps += (10 * torch.log10(1 / m.clamp(min=1e-10))).sum().item()
        ss += ssim(p, y, data_range=1.0, size_average=False).sum().item()
        n += x.size(0)
    G.train()
    return mse / n, ps / n, ss / n          # MSE, PSNR, SSIM

START, best = 0, -1.0
if RESUME and os.path.exists(f"{OUT}/last.pt"):
    ck = torch.load(f"{OUT}/last.pt", map_location=device)
    G.load_state_dict(ck["G"]); D.load_state_dict(ck["D"])
    optG.load_state_dict(ck["optG"]); optD.load_state_dict(ck["optD"])
    START, best = ck["ep"] + 1, ck["best"]
    print("resumed from epoch", START, "| best val SSIM", round(best, 4))

In [ ]:
# 6. Train (best checkpoint by val SSIM, last.pt every epoch)
LOG = f"{OUT}/log.txt"
t0 = time.time()
for ep in range(START, EPOCHS):
    gl = dl_ = 0.0
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        real = torch.ones(x.size(0), dtype=torch.long, device=device)
        fake_lbl = torch.zeros_like(real)

        with autocast("cuda"):
            fake = G(x)

        # D step
        optD.zero_grad(set_to_none=True)
        with autocast("cuda"):
            loss_d = ce(D(y), real) + ce(D(fake.detach()), fake_lbl)
        scD.scale(loss_d).backward(); scD.step(optD); scD.update()

        # G step: (1-delta)*MSE + delta*(1-MS-SSIM) + w*adv
        optG.zero_grad(set_to_none=True)
        with autocast("cuda"):
            adv = ce(D(fake), real)
        f32 = fake.float()
        loss_g = (1 - DELTA) * F.mse_loss(f32, y) + DELTA * (1 - ms_ssim(f32, y, data_range=1.0)) + ADV_W * adv.float()
        scG.scale(loss_g).backward(); scG.step(optG); scG.update()
        gl += loss_g.item(); dl_ += loss_d.item()

    mse, psnr, ssim_v = evaluate(val_dl)
    line = (f"ep{ep:03d} G {gl/len(train_dl):.4f} D {dl_/len(train_dl):.4f} | "
            f"val MSE {mse:.4f} PSNR {psnr:.2f} SSIM {ssim_v:.4f} | {time.time()-t0:.0f}s")
    print(line)
    with open(LOG, "a") as f:
        f.write(line + "\n")

    if ssim_v > best:
        best = ssim_v
        torch.save({"G": G.state_dict(), "D": D.state_dict(), "ep": ep}, f"{OUT}/best.pt")
    torch.save({"G": G.state_dict(), "D": D.state_dict(), "optG": optG.state_dict(),
                "optD": optD.state_dict(), "ep": ep, "best": best}, f"{OUT}/last.pt")

In [ ]:
# 7. Test with best checkpoint
ck = torch.load(f"{OUT}/best.pt", map_location=device)
G.load_state_dict(ck["G"])
mse, psnr, ssim_v = evaluate(test_dl)
print(f"best epoch {ck['ep']} | TEST MSE {mse:.4f} | PSNR {psnr:.2f} | SSIM {ssim_v:.4f}")
pd.DataFrame([{"MSE": mse, "PSNR": psnr, "SSIM": ssim_v}]).to_csv(f"{OUT}/test_metrics.csv", index=False)

In [ ]:
# 8. Save enhanced test images at original size (for NFIQ2 or visuals)
from pathlib import Path
G.eval()
EN = Path(OUT) / "enhanced_test"; EN.mkdir(exist_ok=True)

@torch.no_grad()
def enhance(path):
    im = Image.open(path).convert("L")
    x = torch.from_numpy(np.array(im.resize((SIZE, SIZE), Image.BILINEAR), dtype=np.uint8))
    x = x[None, None].float().div(255).to(device)
    with autocast("cuda"):
        p = G(x)
    p = p.float().clamp(0, 1)[0, 0].cpu().numpy()
    return Image.fromarray((p * 255).round().astype(np.uint8)).resize(im.size, Image.BICUBIC)

test_df = df[df.split == "test"].reset_index(drop=True)
for i, r in test_df.iterrows():
    enhance(r.noisy_path).save(EN / f"{i:04d}_{Path(r.clean_path).stem}.png", dpi=(500, 500))
print("saved", len(list(EN.glob("*.png"))), "images in", EN)